In [1]:
# %load_ext autoreload
# %autoreload 2

In [2]:
# if google colab
use_colab = True

# download packages
!pip install pypose
!pip install kornia

# mount drive with data
from google.colab import drive
drive.mount('/content/drive')

# clone repo
%cd /content/
!git clone https://github.com/MarcinJanis/SonarOdometry.git
%cd SonarOdometry



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content
fatal: destination path 'SonarOdometry' already exists and is not an empty directory.
/content/SonarOdometry


In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.transforms import v2
import numpy as np
import cv2
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import ConnectionPatch
from matplotlib import cm
import os, sys
import pandas as pd

import plotly.graph_objects as go

from box import Box
import yaml

root_dir = os.path.abspath('../..')

if root_dir not in sys.path:
    sys.path.append(root_dir)

from src.data_loader.utils import img_polar2cart
from src.data_loader.transforms import SpeckleNoise, RayArtifacts
from src.data_loader.evaluation_data_generator import DataGenerator


In [4]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch

def visualize_matches(img1, img2, pts1, pts2, conf, n_top=500, inlier_mask=None):
    img1_np = img1.squeeze().cpu().numpy()
    img2_np = img2.squeeze().cpu().numpy()

    pts1_np = pts1.clone().cpu().numpy()
    pts2_np = pts2.clone().cpu().numpy()

    img1_rgb = cv2.cvtColor((img1_np * 255).astype(np.uint8), cv2.COLOR_GRAY2RGB)
    img2_rgb = cv2.cvtColor((img2_np * 255).astype(np.uint8), cv2.COLOR_GRAY2RGB)

    h, w = img1_rgb.shape[:2]
    combined_img = np.concatenate((img1_rgb, img2_rgb), axis=1)

    fig, ax = plt.subplots(figsize=(24, 12), dpi=250, facecolor='black')
    ax.imshow(combined_img)


    best_values, best_n_indices_topk = torch.topk(conf, min(n_top, len(conf)))
    indices_np = best_n_indices_topk.cpu().numpy()

    pts1_sorted = pts1_np[indices_np]
    pts2_sorted = pts2_np[indices_np]


    if inlier_mask is not None:
        if torch.is_tensor(inlier_mask):
            inlier_mask_np = inlier_mask.cpu().numpy()
        else:
            inlier_mask_np = np.array(inlier_mask)
        inlier_mask_sorted = inlier_mask_np[indices_np]
    else:

        inlier_mask_sorted = np.ones(len(pts1_sorted), dtype=bool)

    if len(pts1_sorted) > 0:
        for i, (pt0, pt1) in enumerate(zip(pts1_sorted, pts2_sorted)):

            x0, y0 = float(pt0[0]), float(pt0[1])
            x1, y1 = float(pt1[0]) + float(w), float(pt1[1])

            is_inlier = inlier_mask_sorted[i]


            color = 'lime' if is_inlier else 'red'
            alpha = 0.8 if is_inlier else 0.4
            linewidth = 0.5 if is_inlier else 0.3
            marker_size = 4 if is_inlier else 2


            line_zorder = 2 if is_inlier else 1

            ax.plot([x0, x1], [y0, y1], color=color, alpha=alpha, linewidth=linewidth, zorder=line_zorder)
            ax.scatter(x0, y0, color=color, s=marker_size, zorder=3)
            ax.scatter(x1, y1, color=color, s=marker_size, zorder=3)

    ax.axis('off')
    plt.subplots_adjust(left=0, right=1, top=1, bottom=0)
    plt.show()

In [5]:
# preprocessing

def fls_filter(frame):
        device = frame.device
        b, c, h, w = frame.shape
        # norm torch tensor -> np array
        frame_np = frame.squeeze().detach().cpu().numpy()
        frame_uint8 = np.clip(frame_np * 255.0, 0, 255).astype(np.uint8)
        # median blur
        blured = cv2.medianBlur(frame_uint8, ksize=3)
        # bilateral filter
        bilateral = cv2.bilateralFilter(blured, d=5, sigmaColor=25, sigmaSpace=5)
        # histogram equalization - CLAHE
        clahe = cv2.createCLAHE(clipLimit=1.5, tileGridSize=(8, 8))
        filtered = clahe.apply(bilateral)
        # np array -> norm torch tensor
        filtered_float = filtered.astype(np.float32) / 255.0
        frame_torch = torch.tensor(filtered_float, device=device).unsqueeze(0)
        return frame_torch


In [6]:
if use_colab:
    root_dir = '/content/SonarOdometry'
    data_root_dir =  '/content/drive/MyDrive/Studia/SonarOdometryDataset/SonarOdometryDataset_sample/test_scenarios/seq_3'
else:
    root_dir = 'C:/Users/janis/Projekty/Magisterka/SonarOdometry'
    data_root_dir =  os.path.join(root_dir, 'SonarOdometryDataset/test_scenarios/seq_3')


model_config_pth = os.path.join(root_dir, 'config/model2d.yaml')
sonar_config_pth = os.path.join(root_dir, 'config/sonar.yaml')
# aracati_pth = os.path.join(root_dir, 'data/aracati2017/fls/4000.png')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

with open(model_config_pth, "r") as f:
            model_config = Box(yaml.safe_load(f))

with open(sonar_config_pth, "r") as f:
            sonar_config = Box(yaml.safe_load(f))

from src.models.utils import ExtrinsicsCalib
# extrinsics_calib = ExtrinsicsCalib(T = [sonar_config.position.x, sonar_config.position.y, sonar_config.position.z],
#                                    R = [sonar_config.position.roll, sonar_config.position.pitch, sonar_config.position.yaw],
#                                    device=device)


# from src.data_loader.transforms import SonarDatasetTranforms

fls_resolution = (model_config.input.polar_height, model_config.input.polar_width)

m = 7.250794635405014
transform = v2.Compose([
    RayArtifacts(0.0, 0.03, 0, 10, num_rays=4, probability=1.0),
    v2.GaussianBlur(kernel_size=(7, 5), sigma=(1.0, 2.0)),
    SpeckleNoise(concentration=m, rate=m)])

data_generator = DataGenerator(data_root_dir, device, fls_resolution=fls_resolution, transforms = transform, calibration = None)

r_min = sonar_config.range.min
r_max = sonar_config.range.max
fov = sonar_config.fov.horizontal

In [7]:
# Init LoFTR model
from kornia.feature import LoFTR
match_points = LoFTR(pretrained='outdoor').to(device).eval()

In [8]:
with torch.no_grad():
  image_num = 15
  image_num_offset = 3

  # read images
  t1, frame1, pose_gt1, depth1 = data_generator.get_sample(image_num, return_visu=False, return_depth=True)
  t2, frame2, pose_gt2, depth2 = data_generator.get_sample(image_num + image_num_offset, return_visu=False, return_depth=True)

  # filtration

  frame1_f = fls_filter(frame1.squeeze(0))
  frame2_f = fls_filter(frame2.squeeze(0))


  # sampling grid for polar -> cart transformation
  c, h, w = frame1_f.shape
  b = 1
  out_h, out_w = h, 2 * h
  y = torch.arange(out_h, device=device, dtype=torch.float32)
  x = torch.arange(out_w, device=device, dtype=torch.float32)
  y, x = torch.meshgrid(y, x, indexing='ij')
  x = x - out_w / 2.0
  y = out_h - y

  scale = (r_max - r_min) / out_h
  x_r = x * scale
  y_r = y * scale + r_min
  r = torch.sqrt(x_r**2 + y_r**2)
  theta = torch.atan2(x_r, torch.clamp(y_r, min=1e-5))

  norm_theta = theta / (fov / 2.0)
  norm_r = (r - r_min) / (r_max - r_min) * 2.0 - 1.0

  polar2cart_grid = torch.stack((norm_theta, -norm_r), dim=-1).unsqueeze(0)

  # transform frames to cart
  frame1_c = F.grid_sample(frame1_f.unsqueeze(0), polar2cart_grid, mode='bilinear', padding_mode='zeros', align_corners=True)
  frame2_c = F.grid_sample(frame2_f.unsqueeze(0), polar2cart_grid, mode='bilinear', padding_mode='zeros', align_corners=True)

  # creare mask
  valid_mask = (norm_theta >= -1.0) & (norm_theta <= 1.0) & (norm_r >= -1.0) & (norm_r <= 1.0)
  print(valid_mask.sum())
  mask = valid_mask.unsqueeze(0).expand(b, -1, -1).float()

  frame1_c *= mask
  frame2_c *= mask

tensor(675482, device='cuda:0')


In [9]:
with torch.inference_mode():
  matches = match_points({'image0': frame1_c,
                          'mask0': mask,
                          'image1': frame2_c,
                          'mask1': mask})

pts1, pts2, confidence = matches['keypoints0'], matches['keypoints1'], matches['confidence']




OutOfMemoryError: CUDA out of memory. Tried to allocate 1.27 GiB. GPU 0 has a total capacity of 14.56 GiB of which 907.81 MiB is free. Including non-PyTorch memory, this process has 13.67 GiB memory in use. Of the allocated memory 12.51 GiB is allocated by PyTorch, and 1.04 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [ ]:
# === Statistics ===

# Max possible pts pairs
b, c, h, w = frame1_c.shape
max_pts_full_img = h*w / (8*8) # if img didn't have blind areas
max_pts_real = max_pts_full_img * float(torch.sum(mask.flatten())) / (h*w) # proportionaly to real img area

# Matched points
pts_matched = len(pts1)

print(f'Matched points: {pts_matched}/{max_pts_real} ({pts_matched/max_pts_real})')

# Confidence
print(f'Confidence: mean: {float(torch.mean(confidence))}, std: {float(torch.std(confidence))}, median: {float(torch.median(confidence))}')

In [ ]:
print(frame1_c.shape)
print(mask.shape)

In [ ]:
# Visualisation
visualize_matches(frame1_c, frame2_c, pts1, pts2, confidence, n_top=200)

In [ ]:
M, inlier_mask = cv2.estimateAffinePartial2D(
pts2, pts1, method=cv2.RANSAC,
ransacReprojThreshold=model_config.feature_matching.ransac_thresh,
maxIters=3000,
confidence=0.999
)

if M is not None and inlier_mask is not None:
    inlier_mask = inlier_mask.ravel().astype(bool)
    inliers_abs = int(inlier_mask.sum())
    outliers_abs = len(pts1) - inliers_abs
    inliers_p = inliers_abs / len(pts1) if len(pts1) > 0 else 0.0
    outliers_p = outliers_abs / len(pts1) if len(pts1) > 0 else 0.0
print(f'Points matching parameters:\ninliers: {inliers_abs} ({inliers_p})\outliers: {outliers_abs} ({outliers_p})')